# 分板块研究平均偏度、峰度影响：中小板

In [1]:
## 导入NumPy、Pandas、Wind
import numpy as np
import pandas as pd
from scipy import stats
from WindPy import w
import datetime
from dateutil.relativedelta import relativedelta

In [2]:
## 读入个股收益率各月的方差、偏度、峰度
prdct0 = pd.read_csv('Raw Transaction Data/prdct_dt.csv')
## 读入上市公司代码与上市板块
board = pd.read_csv('Raw Transaction Data/AllCorps.csv')
## 合并指标值与板块信息
prdct_mk = pd.merge(prdct0, board, left_on = 'code', right_on = 'codes', how = 'left')

In [3]:
## 启动Wind终端的API
## 判断WindPy是否已经登录成功
w.start() 
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2020 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [4]:
## 读取shibor数据
shibor = w.wsd("SHIBORON.IR", "open", "2007-01-01", "2019-01-31")
shibor = pd.DataFrame(shibor.Data,index = shibor.Codes,columns = shibor.Times)
shibor = shibor.T
shibor = shibor.reset_index()
shibor.columns = ['date', 'shibor']
shibor['shibor'] = shibor['shibor'] * 0.01
shibor['date'] = pd.to_datetime(shibor['date'])
shibor['year'] = shibor['date'].dt.year
shibor['month'] = shibor['date'].dt.month

In [5]:
## 计算各月shibor均值
intrst = shibor.groupby([shibor['year'], shibor['month']])['shibor'].agg([('mn_SHIBOR', 'mean'), ])
intrst = intrst.reset_index()
intrst = intrst.reset_index()
intrst.columns = ['index', 'year', 'month', 'shibor']

In [6]:
## 保留创业板指标
prdct = prdct_mk[prdct_mk['boardtype'] == '创业板']
prdct = prdct.reset_index(drop = True)
prdct = prdct.sort_values(by= ["code", 'year', 'month'] , ascending = True)

In [7]:
codes = prdct['code'].tolist()
codes = list(set(codes))
len(codes)

738

## 计算领先一期的超额收益率

In [8]:
## 指数月收益数据
mkt_r = w.wsd("399102.SZ", "pct_chg", "2007-01-01", "2019-01-31", "Period=M;TradingCalendar=SZSE")
mkt_r = pd.DataFrame(mkt_r.Data, index = mkt_r.Codes, columns = mkt_r.Times)
mkt_r = mkt_r.T
mkt_r = mkt_r.reset_index()
mkt_r.columns = ['date', 'pct_chg']
mkt_r['date'] = pd.to_datetime(mkt_r['date'])
mkt_r['pct_chg'] = mkt_r['pct_chg'] * 0.01
mkt_r = mkt_r.reset_index()

In [9]:
## 计算月超额收益率
mkt_exr = pd.merge(mkt_r, intrst, on = 'index', how = 'left')
mkt_exr['exr'] = mkt_exr['pct_chg'] - mkt_exr['shibor']
mkt_exr = mkt_exr[['index', 'exr', 'year', 'month', 'date']]

In [10]:
mkt_r0 = mkt_exr.head(144)
mkt_r0.columns = ['index', 'r0', 'year', 'month', 'date']

In [11]:
## 删除首行，得到领先一期的收益率
mkt_exr = mkt_exr[['exr', 'year', 'month']]
mkt_exr = mkt_exr.drop(0, axis = 0)
mkt_exr = mkt_exr.reset_index(drop = True)
mkt_exr = mkt_exr.reset_index()

## 计算市场方差、偏度、峰度

In [12]:
## 获得创业板综指的指数数据
trdt_mk = w.wsd("399102.SZ", "swing", "2007-01-01", "2019-01-31", "TradingCalendar=SZSE")
trdt_mk = pd.DataFrame(trdt_mk.Data, index = trdt_mk.Codes, columns = trdt_mk.Times)
trdt_mk = trdt_mk.T
trdt_mk = trdt_mk.reset_index()
trdt_mk.columns = ['date', 'pct_chg']
trdt_mk['pct_chg'] = trdt_mk['pct_chg'] * 0.01
trdt_mk['date'] = pd.to_datetime(trdt_mk['date'])
trdt_mk['year'] = trdt_mk['date'].dt.year
trdt_mk['month'] = trdt_mk['date'].dt.month

In [13]:
## 合并中小板综指数据和shibor收益数据
mkt_dt = pd.merge(trdt_mk, shibor[['date', 'shibor']], on = 'date', how = 'left')
mkt_dt['exr'] = mkt_dt['pct_chg'] - mkt_dt['shibor']
mkt_dt = mkt_dt[['year', 'month', 'date', 'pct_chg', 'shibor', 'exr']]

In [14]:
## 分组，计算各月均值、方差、偏度、峰度
mkt_sta = mkt_dt.groupby([mkt_dt['year'], mkt_dt['month']])['exr'].agg([('num', 'count'), ('mean', 'mean'), ('math_var', 'var'), ('math_skew', 'skew'),('math_kurt', stats.kurtosis),])
mkt_sta = mkt_sta.reset_index()

In [15]:
## 删除首行
mkt_lf = mkt_dt.groupby([mkt_dt['year'], mkt_dt['month']])['date'].first()
mkt_lf = mkt_lf.reset_index()
mkt_lf.insert(2, 'flag', np.ones(len(mkt_lf)))
mkt_lf = mkt_lf[['date', 'flag']]

## 保留其他数据
mkt_wof = pd.merge(mkt_dt, mkt_lf, on = 'date', how = 'left')
mkt_wof = mkt_wof[mkt_wof['flag'] != 1]
mkt_wof = mkt_wof[['date', 'exr']]
mkt_wof = mkt_wof.rename(columns={'exr': 'exr_f'})
mkt_wof.insert(2, 'order', np.arange(len(mkt_wof)))

## 删除末行
mkt_ll = mkt_dt.groupby([mkt_dt['year'], mkt_dt['month']])['date'].last()
mkt_ll = mkt_ll.reset_index()
mkt_ll.insert(2, 'flag', np.ones(len(mkt_ll)))
mkt_ll = mkt_ll[['date', 'flag']]

## 保留其他数据
mkt_wol = pd.merge(mkt_dt, mkt_ll, on = 'date', how = 'left')
mkt_wol = mkt_wol[mkt_wol['flag'] != 1]
mkt_wol = mkt_wol[['year', 'month', 'exr']]
mkt_wol = mkt_wol.rename(columns={'exr': 'exr_l'})
mkt_wol.insert(2, 'order', np.arange(len(mkt_wol)))

## 合并去除第一行/最后一行的数据，准备计算调整的方差
## 注意，保留的日期系exr_f的日期
mkt_ajsta0 = pd.merge(mkt_wol, mkt_wof, on = 'order', how = 'left')
mkt_ajsta0 = mkt_ajsta0[[ 'year', 'month', 'date', 'exr_f', 'exr_l']]

## 合并均值数据
mkt_ajsta = pd.merge(mkt_ajsta0, mkt_sta[['year', 'month', 'mean']], on = ['year', 'month'], how = 'left')
mkt_ajsta['diff1'] = mkt_ajsta['exr_f'] - mkt_ajsta['mean']
mkt_ajsta['diff2'] = mkt_ajsta['exr_l'] - mkt_ajsta['mean']
mkt_ajsta['product'] = mkt_ajsta['diff1'] * mkt_ajsta['diff2']
mkt_ajsta = mkt_ajsta.groupby([mkt_ajsta['year'], mkt_ajsta['month']])['product'].agg([('prdct', 'sum'),])
mkt_ajsta = mkt_ajsta.reset_index()

## 合并调整后的数据和原统计指标值
mkt_ajs = pd.merge(mkt_sta, mkt_ajsta, on = ['year', 'month'], how = 'left' )
mkt_ajs['var'] = mkt_ajs['math_var'] * mkt_ajs['num'] + 2* mkt_ajs['prdct']
mkt_ajs['adj_var'] = mkt_ajs['var']/mkt_ajs['num']

## 保留数据
mkt_prdct = mkt_ajs[['year', 'month', 'num', 'mean', 'var', 'adj_var', 'math_skew', 'math_kurt']]
mkt_prdct = mkt_prdct[['year', 'month', 'var', 'math_skew', 'math_kurt']]
mkt_prdct.columns = ['year', 'month', 'mkt_var', 'mkt_skew', 'mkt_kurt']

In [16]:
mkt_prdct

,year,month,mkt_var,mkt_skew,mkt_kurt
0,2007,1,NaN,NaN,NaN
1,2007,2,NaN,NaN,NaN
2,2007,3,NaN,NaN,NaN
3,2007,4,NaN,NaN,NaN
4,2007,5,NaN,NaN,NaN
...,...,...,...,...,...
140,2018,9,0.000406,-0.377135,-1.260054
141,2018,10,0.002534,0.606260,-0.996331
142,2018,11,0.001338,1.084392,0.128041
143,2018,12,0.001712,0.952776,0.136288


## 等权重加权法

In [17]:
## 计算创业板平均方差、偏度与峰度
mnv_ew = prdct.groupby([prdct['year'], prdct['month']])['var', 'adj_var', 'math_skew', 'math_kurt'].mean()
mnv_ew = mnv_ew.reset_index()
mnv_ew = mnv_ew[['year', 'month', 'adj_var', 'math_skew', 'math_kurt']]
mnv_ew.columns = ['year', 'month', 'var_ew', 'skew_ew', 'kurt_ew']

In [18]:
mnv_ew

,year,month,var_ew,skew_ew,kurt_ew
0,2009,11,0.002639,-0.510508,-0.108868
1,2009,12,0.001457,-0.191957,0.156872
2,2010,1,0.001897,0.791106,1.668415
3,2010,2,0.001022,0.013159,0.437106
4,2010,3,0.000723,-0.059888,0.096379
...,...,...,...,...,...
105,2018,8,0.000861,0.271952,0.239175
106,2018,9,0.000535,-0.039269,0.448894
107,2018,10,0.001494,-0.507425,0.645489
108,2018,11,0.000980,-0.124778,0.820701


## 市值加权法

In [19]:
## 读取市值数据
evdt = w.wsd(codes, "ev", "2006-12-01", "2018-11-30", "unit=1;Period=M;TradingCalendar=SZSE")
ev = pd.DataFrame(evdt.Data, index = evdt.Codes, columns = evdt.Times)
ev = ev.T
ev = ev.reset_index()

In [20]:
## 转为一维表，准备合并
ev_T = pd.melt(ev, 
            id_vars = 'index', 
            value_vars = list(ev.columns[1:]), 
            var_name='code', 
            value_name='ev_month')
ev_T.columns = ['date', 'code', 'ev_month']
ev_T['date'] = pd.to_datetime(ev_T['date'])
ev_T['date'] = ev_T['date'].apply(lambda x: x + relativedelta(months = +1)) 
## 生成年、月
ev_T['year'] = ev_T['date'].dt.year
ev_T['month'] = ev_T['date'].dt.month

In [21]:
## 合并prdct和ev_w
vw_fore = pd.merge(prdct, ev_T, on = ['code', 'year', 'month'], how = 'left')
vw_fore = vw_fore.dropna().reset_index(drop = True)

## 按月分组，计算各月总市值
## 计算样本数据的权重
tt_ev = vw_fore.groupby([vw_fore['year'], vw_fore['month']])['ev_month'].agg([('TEV', 'sum'),])
tt_ev = tt_ev.reset_index()
ev_w = pd.merge(vw_fore, tt_ev, on = ['year', 'month'], how = 'left')
ev_w['weight'] = ev_w['ev_month']/ ev_w['TEV']
ev_w = ev_w.sort_values(by= ["code", 'year', 'month'] , ascending = True)

In [22]:
ev_w

,code,year,month,num,var,adj_var,math_skew,math_kurt,codes,boardtype,date,ev_month,TEV,weight
0,300001.SZ,2009,11,21,0.052577,0.002504,-0.731528,-0.141233,300001.SZ,创业板,2009-11-30,5.878400e+09,1.399674e+11,0.041998
1,300001.SZ,2009,12,23,0.020632,0.000897,-0.396938,-0.372213,300001.SZ,创业板,2009-12-30,6.268512e+09,1.445010e+11,0.043380
2,300001.SZ,2010,1,20,0.014681,0.000734,0.299213,0.625364,300001.SZ,创业板,2010-01-31,5.643264e+09,1.610084e+11,0.035050
3,300001.SZ,2010,2,15,0.025158,0.001677,-0.833630,0.187893,300001.SZ,创业板,2010-02-28,4.864376e+09,1.843256e+11,0.026390
4,300001.SZ,2010,3,23,0.026632,0.001158,0.608117,0.721363,300001.SZ,创业板,2010-03-26,5.263840e+09,2.324507e+11,0.022645
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44549,300750.SZ,2018,12,20,0.004244,0.000212,0.139163,-0.120247,300750.SZ,创业板,2018-12-30,1.756672e+11,4.239045e+12,0.041440
44550,300751.SZ,2018,12,20,0.015891,0.000795,-0.599014,-0.276630,300751.SZ,创业板,2018-12-30,6.644560e+09,4.239045e+12,0.001567
44551,300752.SZ,2018,12,20,0.161407,0.008070,0.066487,-1.607825,300752.SZ,创业板,2018-12-30,2.183469e+09,4.239045e+12,0.000515
44552,300760.SZ,2018,11,22,0.003906,0.000178,1.308318,2.043055,300760.SZ,创业板,2018-11-30,1.203534e+11,4.016866e+12,0.029962


In [23]:
## 筛选需要的指标列
vw = ev_w[['code', 'year', 'month', 'weight', 'var', 'adj_var', 'math_skew', 'math_kurt']]

## 计算加权指标
vw['var_vw'] = vw['adj_var'] * vw['weight']
vw['skew_vw'] = vw['math_skew'] * vw['weight']
vw['kurt_vw'] = vw['math_kurt'] * vw['weight']

## 按月加总
mnv_vw = vw.groupby([vw['year'], vw['month']])['var_vw', 'skew_vw', 'kurt_vw'].sum()
mnv_vw = mnv_vw.reset_index()
mnv_vw = mnv_vw[['year', 'month', 'var_vw', 'skew_vw', 'kurt_vw']]

## 合并数据

In [24]:
mkt_reg = pd.merge(mkt_exr[['index', 'exr']], mkt_r0, on = 'index', how = 'left')
mkt_reg = pd.merge(mkt_reg, mkt_prdct, on = ['year', 'month'], how = 'left')
mkt_reg = pd.merge(mkt_reg, mnv_ew, on = ['year', 'month'], how = 'left')
mkt_reg = pd.merge(mkt_reg, mnv_vw, on = ['year', 'month'], how = 'left')
mkt_reg = mkt_reg.drop(['index', 'year', 'month'], axis = 1)
mkt_reg = mkt_reg.dropna()
mkt_reg.columns = ['r', 'r0', 'date','mkt_var', 'mkt_skew', 'mkt_kurt', 'var_ew', 'skew_ew', 'kurt_ew', 'var_vw', 'skew_vw', 'kurt_vw']
mkt_reg = mkt_reg.reset_index(drop = True)

In [25]:
mkt_reg[mkt_reg['date'] < '2015-01-01'].to_csv("Plot/reg_mkt3.csv", index = False) 

In [26]:
mkt_reg[mkt_reg['date'] < '2015-01-01']

,r,r0,date,mkt_var,mkt_skew,mkt_kurt,var_ew,skew_ew,kurt_ew,var_vw,skew_vw,kurt_vw
0,0.036843,-0.102268,2010-06-30,0.005363,0.548418,-0.471148,0.002085,-0.298269,-0.310924,0.002204,-0.288895,-0.243211
1,0.042345,0.036843,2010-07-30,0.003642,0.710287,-0.458944,0.000829,-0.185315,0.323053,0.000849,-0.167537,0.095654
2,-0.100472,0.042345,2010-08-31,0.000766,0.431462,0.616646,0.002605,0.107612,0.782267,0.000546,-0.070916,0.156086
3,0.113241,-0.100472,2010-09-30,0.001503,0.763375,0.028566,0.001933,0.103414,0.249038,0.000758,-0.031506,-0.182028
4,0.081943,0.113241,2010-10-29,0.005150,2.901298,6.556917,0.001117,0.642776,0.417207,0.000925,0.562810,0.318343
5,-0.065886,0.081943,2010-11-30,0.007173,0.567322,-0.857830,0.002072,0.087882,0.142714,0.001128,-0.077565,-0.290992
6,-0.139407,-0.065886,2010-12-31,0.008646,-0.087684,-0.225593,0.001750,0.027841,0.250553,0.001379,-0.143416,0.235527
7,0.039703,-0.139407,2011-01-31,0.027992,-0.541770,-0.803451,0.001689,-0.216549,-0.118820,0.001540,-0.289460,-0.193060
8,-0.094045,0.039703,2011-02-28,0.002427,0.060195,-0.678176,0.000741,0.073371,0.026075,0.000405,-0.003552,-0.135723
9,-0.114261,-0.094045,2011-03-31,0.001197,0.927711,-0.085306,0.000585,-0.152789,0.825409,0.000510,-0.183427,0.637449
